In [1]:
!pip install transformers

In [2]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [3]:
classifier("I love this!")

[[{'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.00041385178337804973},
  {'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'neutral', 'score': 0.005764591973274946},
  {'label': 'sadness', 'score': 0.0020923891570419073},
  {'label': 'surprise', 'score': 0.00852868054062128}]]

In [5]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",  # path to your model
    framework="tf"              # forces use of TensorFlow
)

print(classifier("I feel great today!"))


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tf_model.h5:   0%|          | 0.00/329M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.
Device set to use 0


[{'label': 'joy', 'score': 0.9926296472549438}]


In [49]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification

# Load the model from the Hugging Face model hub (TensorFlow version)
model = TFAutoModelForSequenceClassification.from_pretrained("j-hartmann/emotion-english-distilroberta-base")

# Save it in the SavedModel format (required for TFLite conversion)
saved_model_path = "/content/saved_model2"
model.save(saved_model_path)

# Convert the model to TFLite with float16 optimization
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Optimize for size
converter.target_spec.supported_types = [tf.float16]  # Use float16 for quantization

# Convert and save the TFLite model
tflite_model = converter.convert()
tflite_model_path = "/content/emotion_analysis_model_2_fp16.tflite"

with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved at: {tflite_model_path}")


All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


TFLite model saved at: /content/emotion_analysis_model_2_fp16.tflite


In [52]:
import tensorflow as tf
import numpy as np
from transformers import AutoTokenizer, AutoConfig

# 0) Load the id2label mapping directly (keys are integers, not strings)
config = AutoConfig.from_pretrained("tuhanasinan/emotion-analysis-with-tensorflow")
labels = [config.id2label[i] for i in range(len(config.id2label))]

# 1) Load your tokenizer (must match the model you converted)
tokenizer = AutoTokenizer.from_pretrained("tuhanasinan/emotion-analysis-with-tensorflow")

# 2) Load your TFLite model
interpreter = tf.lite.Interpreter(model_path="/content/emotion_analysis_model_fp16.tflite")

# 3) Prepare some example text
text = "you are hurting me"

# 4) Tokenize to NumPy arrays, pad/truncate to the same max_length you used for conversion
enc = tokenizer(
    text,
    return_tensors="np",
    padding="max_length",
    truncation=True,
    max_length=512
)
input_ids      = enc["input_ids"]
attention_mask = enc["attention_mask"]

# 5) Inspect interpreter I/O
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 6) Resize each input to match your tokenized arrays
for d in input_details:
    if "input_ids" in d["name"]:
        interpreter.resize_tensor_input(d["index"], input_ids.shape)
    elif "attention_mask" in d["name"]:
        interpreter.resize_tensor_input(d["index"], attention_mask.shape)
interpreter.allocate_tensors()

# 7) Populate input tensors (cast to the expected dtype)
for d in input_details:
    if "input_ids" in d["name"]:
        arr = input_ids.astype(d["dtype"])
    else:
        arr = attention_mask.astype(d["dtype"])
    interpreter.set_tensor(d["index"], arr)

# 8) Run the model
interpreter.invoke()

# 9) Retrieve logits and compute softmax probabilities
logits = interpreter.get_tensor(output_details[0]["index"])
probs  = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)

# 10) Decode top prediction
pred = np.argmax(probs, axis=-1)[0]
print(f"Prediction: {labels[pred]} ({probs[0, pred]:.4f})")


Prediction: sadness (0.9932)


In [55]:
import tensorflow as tf
import numpy as np
from transformers import AutoTokenizer, AutoConfig

# 0) Load id2label mapping
config = AutoConfig.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
labels = [config.id2label[i] for i in range(len(config.id2label))]

# 1) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")

# 2) Load TFLite interpreter
interpreter = tf.lite.Interpreter(model_path="/content/emotion_analysis_model_2_fp16.tflite")

# 3) Tokenize your input
text = "I'm absolutely thrilled with how things turned out!"
enc = tokenizer(
    text,
    return_tensors="np",
    padding="max_length",
    truncation=True,
    max_length=512
)
input_ids      = enc["input_ids"]      # shape (1, 512)
attention_mask = enc["attention_mask"] # shape (1, 512)

# 4) Grab I/O details
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 5) Resize _all_ inputs to (1, 512)
new_shape = input_ids.shape
for d in input_details:
    interpreter.resize_tensor_input(d["index"], new_shape)
interpreter.allocate_tensors()

# 6) Feed in data (or zeros if we don’t have a name match)
for d in input_details:
    name = d["name"]
    if "input_ids" in name:
        arr = input_ids
    elif "attention_mask" in name:
        arr = attention_mask
    else:
        # any extra input → fill with zeros
        arr = np.zeros(new_shape, dtype=d["dtype"])
    interpreter.set_tensor(d["index"], arr.astype(d["dtype"]))

# 7) Inference
interpreter.invoke()

# 8) Read logits & softmax
logits = interpreter.get_tensor(output_details[0]["index"])
probs  = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)

# 9) Decode
pred = np.argmax(probs, axis=-1)[0]
print(f"Prediction: {labels[pred]} ({probs[0, pred]:.4f})")

# 10) (Optional) Dump full distribution
for lab, p in zip(labels, probs[0]):
    print(f"{lab:8s}: {p:.4f}")


Prediction: joy (0.8546)
anger   : 0.0120
disgust : 0.0017
fear    : 0.0020
joy     : 0.8546
neutral : 0.0223
sadness : 0.0022
surprise: 0.1052
